# 05 — Conclusiones

**Objetivo:** Cerrar el proyecto. Resumir qué se descubrió en todo el proceso, qué problemas presentaban los datos (limitaciones) y qué se mejoraría a futuro.

Este notebook es principalmente narrativo. Carga el log de auditoría y el dataset limpio solo para respaldar las conclusiones con números.

## Respaldo: log de auditoría y estado final

In [1]:
import pandas as pd

log = pd.read_csv('../logs/pipeline_log.csv')
df = pd.read_csv('../data/processed/streaming_users_clean.csv')

print("=== Recorrido del pipeline de limpieza ===")
print(log.to_string(index=False))
print(f"\nDataset final: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Retención estructural: {log['Retención (%)'].iloc[-1]}%")

=== Recorrido del pipeline de limpieza ===
 Paso                           Descripción  Filas  Nulos  Retención (%)
    0                       Estado original   8160    753         100.00
    1                   Eliminar duplicados   8000    753          98.04
    2                Normalizar categóricas   8000    753          98.04
    3            Valores imposibles -> nulo   8000   1049          98.04
    4                Parsear/validar fechas   8000   1128          98.04
    5 Winsorizar watch_time (k=3, lim=2693)   8000   1128          98.04
    6     Imputar faltantes (excepto fecha)   8000    399          98.04

Dataset final: 8000 filas, 8 columnas
Retención estructural: 98.04%


## 1. Qué descubrimos

**Sobre la calidad de los datos (notebooks 01–02):**
- El dataset crudo tenía problemas en las 8 columnas: 160 duplicados, valores imposibles (edades negativas, tiempos negativos), valores imposibles de relleno (99999, 99, 150), múltiples grafías de las mismas categorías y una columna de fechas con formatos mezclados y basura.
- Tras la limpieza se conservó cerca del **98%** de las filas; la pérdida se concentró únicamente en duplicados documentados.

**Sobre el comportamiento de los usuarios (notebook 03 – EDA):**
- La base es de **adultos jóvenes** (mediana ~33 años).
- El **plan de suscripción es el principal determinante del consumo**: los usuarios Premium miran casi el doble que los Básico.
- La **edad no influye** en cuánto mira una persona.
- La preferencia de **género es transversal** al plan y al país; el mercado es geográficamente homogéneo.

**Sobre la forma de las variables (notebook 03 – EDA):**
- El consumo y los tickets presentan **asimetría positiva alta** y el consumo es **leptocúrtico** (colas pesadas por los super-usuarios). El coeficiente de variación muestra que los tickets son la variable más heterogénea (~111%) y la edad la más homogénea (~34%). Esto respalda el uso de la mediana sobre la media.
- El diagnóstico del mecanismo de faltantes mostró que los nulos de watch_time crecen con el plan (**MAR**), justificando la imputación por grupo.

**Sobre la estructura de las variables (notebook 04 – PCA):**
- Las variables numéricas (`age`, `watch_time`, `tickets`) son **estadísticamente independientes**: cada componente principal explica ~33% de la varianza, así que **no es posible reducir dimensiones sin perder información**. PCA funcionó como diagnóstico, confirmando ausencia de redundancia.

## 2. Limitaciones y problemas encontrados

- **Origen probablemente sintético:** la independencia casi total entre variables (correlaciones ≈ 0, PCA sin compresión, categóricas sin asociación) sugiere que el dataset fue generado artificialmente con variables poco relacionadas. En datos reales esperaríamos más estructura. Esto limita el alcance de los hallazgos de relación.
- **Valores imposibles vs. outliers:** distinguir un valor imposible de relleno (99999) de un outlier real exigió criterio. Decidimos por evidencia (huecos en la distribución y repetición exacta), pero es una decisión interpretable que otro analista podría discutir.
- **Imputación:** rellenar nulos con la mediana por plan introduce cierto sesgo hacia el centro (reduce la varianza real). Es un costo asumido a cambio de no perder filas.
- **`last_login_date` incompleta:** quedaron ~400 fechas nulas que decidimos **no** imputar para no inventar datos. Esto limita cualquier análisis de actividad temporal o retención basado en el último acceso.
- **Sin variable objetivo:** el dataset no trae una etiqueta (ej. "se dio de baja"), así que no se pudo evaluar capacidad predictiva real ni hacer un modelo supervisado.

## 3. Qué mejoraríamos a futuro

- **Validación en origen:** aplicar reglas de carga (rangos válidos de edad, formato único de fecha, catálogo cerrado de planes/países/géneros) evitaría gran parte de la limpieza posterior.
- **Imputación más sofisticada:** probar métodos como KNN o imputación múltiple en lugar de la mediana, y comparar el impacto en la distribución.
- **Enriquecer los datos:** incorporar una variable objetivo (baja/retención) y datos de actividad temporal permitiría pasar del análisis descriptivo a uno predictivo.
- **Análisis temporal:** si se recuperaran las fechas de login, estudiar patrones de uso a lo largo del tiempo y estacionalidad.
- **Segmentación (clustering):** aunque PCA no mostró estructura clara, probar algoritmos de agrupamiento sobre las variables escaladas para confirmar si existen segmentos de usuarios no evidentes.
- **Automatizar el pipeline:** convertir los notebooks 01–02 en un script reproducible que registre el log automáticamente ante nuevos lotes de datos.

## Cierre

El proyecto recorrió el ciclo completo de preparación de datos: **inspección → limpieza documentada → análisis exploratorio → reducción de dimensiones → conclusiones**. El resultado es un dataset limpio, auditable y comprendido, con cada decisión justificada por evidencia y con sus limitaciones explicitadas. El mayor aprendizaje metodológico fue que un análisis honesto incluye reportar tanto lo que los datos muestran como lo que **no** muestran (ausencia de relaciones, posible origen sintético), en lugar de forzar conclusiones.